In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import  train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

In [2]:
df = pd.read_csv('credit_risk_dataset.csv')
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [3]:
numeric_col = [
    'person_age',
    'person_income',
    'person_emp_length',
    'loan_amnt',
    'loan_int_rate',
    'loan_percent_income',
    'cb_person_cred_hist_length',
]

categorical_cols = [
    'person_home_ownership',
    'loan_intent',
    'loan_grade',
    'cb_person_default_on_file',
]


In [10]:
X_num = df[numeric_col].copy()
X_num = X_num.fillna(X_num.mean())

X_cat = pd.get_dummies(df[categorical_cols], drop_first = True)

X = pd.concat([X_num, X_cat], axis=1)
feature_cols = X.columns.tolist()
y = df['loan_status'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [11]:
from IPython.lib.security import random
models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'K-Nearest Neighbors': make_pipeline(StandardScaler(), KNeighborsClassifier())
}

best_f1 = -1
best_model_name = ""
best_model_obj  = None

print("Modellar oqitilmoqda")
print(f"{'Model Nomi':<22} | {'Accuracy':<10} | {'F1 (default)':<14} | {'ROC-AUC'}")


for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred) * 100
    f1 = f1_score(y_test, y_pred) * 100
    auc = roc_auc_score(y_test, y_proba) * 100

    print(f"{name:<22} | {acc:>7.2f}%  | {f1:>11.2f}%  | {auc:.2f}%")


    if f1 > best_f1:
        best_f1 = f1
        best_model_name = name
        best_model_obj = model

print("-" * 65)
print(f"\n🏆 ENG YAXSHI MODEL: {best_model_name} (F1={best_f1:.2f}%)")

print(f"\nBatafsil hisobot ({best_model_name}):")
print(classification_report(y_test, best_model_obj.predict(X_test), target_names=["No Default", "Default"]))

Modellar oqitilmoqda
Model Nomi             | Accuracy   | F1 (default)   | ROC-AUC
Logistic Regression    |   86.77%  |       65.04%  | 86.94%
Decision Tree          |   88.45%  |       74.13%  | 83.92%
Random Forest          |   93.28%  |       82.42%  | 93.06%
Gradient Boosting      |   92.88%  |       81.38%  | 93.50%
K-Nearest Neighbors    |   89.37%  |       71.75%  | 85.30%
-----------------------------------------------------------------

🏆 ENG YAXSHI MODEL: Random Forest (F1=82.42%)

Batafsil hisobot (Random Forest):
              precision    recall  f1-score   support

  No Default       0.93      0.99      0.96      5095
     Default       0.96      0.72      0.82      1422

    accuracy                           0.93      6517
   macro avg       0.94      0.86      0.89      6517
weighted avg       0.93      0.93      0.93      6517



In [13]:
clean_name = best_model_name.lower().replace(" ", "_")
model_file = f"models/{clean_name}_model.joblib"

print("\n3. Eng yaxshi model faylga saqlanmoqda...")
os.makedirs("models", exist_ok=True)
joblib.dump(best_model_obj, model_file)
joblib.dump(feature_cols, 'models/credit_features.joblib')

print("\n🚀 HAMMASI TAYYOR!")
print(f"--- G'olib model '{model_file}' nomi bilan saqlandi.")
print("--- Ustunlar ro'yxati 'models/credit_features.joblib' nomi bilan saqlandi.")



3. Eng yaxshi model faylga saqlanmoqda...

🚀 HAMMASI TAYYOR!
--- G'olib model 'models/random_forest_model.joblib' nomi bilan saqlandi.
--- Ustunlar ro'yxati 'models/credit_features.joblib' nomi bilan saqlandi.
